# The byte screen — 2,000-step continuation arms of the softmax twin and of the splat 2s

**What this is.** The byte battery (reports/byte_battery_2026-10-10 on the training repo) read nine stored checkpoints bit
by bit and found that no model keys a lookup among competing records, that the splat crafts never fetch a planted value
at its cue behind text, and that the loss profile is the text's (88% in the identity bits for every model). The levers
that follow from that are data (far-recall rows rendered into fineweb text) and, as cheap controls, two reweighted
losses. This notebook runs them as 2,000-step continuation arms from a stored checkpoint, one side per session:

| side | source checkpoint | arms (`KINDS`) | micro-batch × accumulation | pace |
|---|---|---|---|---|
| `twin` | the fp16 softmax twin resumed from 10,000, at `START_STEP` (16,000 or 18,000) | `ctrl` the bit-identical control · `rows` the far-recall rows at 5% in place of fineweb text · `rows-copy` the rows + the copy-weighted loss (β 2) · `depth` the depth-weighted loss (1,1,1,2,2,2,2,2 at mean 1) | 8 × 8, fp16 flash in the fp32 attention block | ~3.1 s a step on an H100 |
| `2s` | the splat 2s at 18,000 | `ctrl` · `rows` · `rows-copy` | 16 × 4 (the 2s recipe) | ~2.4 s a step |

Every arm trains the recipe's 262,144-token step and reads the same rows per step (one seed offset), so the arms differ
by their lever alone. The rows kinds start inside the fineweb phase (`START_STEP` ≤ 18,219), so the rows replace web
text and nothing else. Each arm is its own craft name = its own prefix on the training repo; its checkpoints and a
report ship as it goes, and the reads (the battery and the probes) run on the arm's shipped checkpoints afterwards.

**The bars** (from the battery): the first byte of a planted value at 1,024 and 2,048 bytes; the four-locker keyed
lookup above chance (.25); the worded same-key update toward the last value; the repeated digit string's position 12;
held-out bits per byte not worse than the control's.

In [ ]:
# C1 — pinned install. RESTART RUNTIME if the version line below changes.
import os
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ.setdefault('OMP_NUM_THREADS', '16')
%pip install -q "datasets>=5.0.0" "geolip-alephllm[train] @ git+https://github.com/AbstractEyes/alephllm@v0.10.15"
import geolip.alephllm as A
print('geolip.alephllm', A.__version__)
assert tuple(int(x) for x in A.__version__.split('.')) >= (0, 10, 15), (
    'RESTART REQUIRED: runtime still holds the old package — Runtime > Restart, then rerun from C1')

from geolip.alephllm.presets import make_byte_screen_preset, BYTE_SCREEN_KINDS

SIDE = 'twin'                 # 'twin' (4 arms, ~7 h on an H100) or '2s' (3 arms, ~4 h)
START_STEP = 16000            # twin: 16000 or 18000 (both battery-read); 2s: 18000
KINDS = ['ctrl', 'rows', 'rows-copy', 'depth'] if SIDE == 'twin' else ['ctrl', 'rows', 'rows-copy']
STEPS_PER_ARM = 2000
HOURS_PER_ARM = None          # None: STEPS_PER_ARM at the pace C2 measures (+5%); or a number of hours
MICRO = None                  # None: the side's recipe (twin 8 x 8, 2s 16 x 4); a smaller card: 4 or 2 (accumulation rises to keep the step)

ARMS = [make_byte_screen_preset(SIDE, k, START_STEP) for k in KINDS]
if MICRO:
    for a in ARMS:
        a.train.grad_accum = a.train.micro_batch * a.train.grad_accum // MICRO
        a.train.micro_batch = MICRO
for a in ARMS:
    fw = [ph for ph in a.curriculum if ph['name'] == 'fineweb_main'][0]['dataset']
    print(f"{a.model.name:48s} loss {a.train.loss_form:10s} fineweb_main -> {fw:22s} {a.train.micro_batch} x {a.train.grad_accum}  from {a.init_from['prefix']}@{a.init_from['step']}")

In [ ]:
# C2 (P) — PREFLIGHT on the first arm: build, the source weights, the step-1 read, the bench (sets HOURS_PER_ARM), VRAM gate.
import time, torch
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from geolip.alephllm.model.alephlm import AlephLM
from geolip.alephllm.data.tokenizers import build_tokenizer
from geolip.alephllm.data.streams import build_stream
from geolip.alephllm.train.precision import autocast
def hf_token():
    import os
    t = os.environ.get('HF_TOKEN')
    if not t:
        from google.colab import userdata
        t = userdata.get('HF_TOKEN')
    assert t, 'no HF token: set HF_TOKEN in the environment (a pod) or in Colab secrets'
    return t

p = ARMS[0]; cfg, tc, src = p.model, p.train, p.init_from
tf32 = tc.precision != 'fp32'
torch.backends.cuda.matmul.allow_tf32 = tf32; torch.backends.cudnn.allow_tf32 = tf32
torch.manual_seed(tc.seed)
model = AlephLM(cfg).cuda()
ck = hf_hub_download(src['repo'], f"{src['prefix']}/{src['path']}", token=hf_token())
sd = {k: v.float() for k, v in load_file(ck).items()}
res = model.load_state_dict(sd, strict=False)
assert not res.unexpected_keys and all(k.endswith(('q_gain', 'k_gain')) for k in res.missing_keys), 'the checkpoint does not fit the craft'
print(f"{cfg.name}: {model.param_count()/1e6:.1f}M params | kernel {cfg.attn_kernel!r} attn_fp32 {cfg.attn_fp32} qk_norm {cfg.qk_norm!r} | from {src['prefix']}/{src['path']}")

# the step-1 read: one real fineweb batch under the arm's precision (the trainer's held-out gauge is the real series)
tok = build_tokenizer(cfg.tokenizer)
x = build_stream('fineweb-edu', tok, cfg.context, 2, seed=tc.seed + 99).next_batch().cuda()
model.eval()
with torch.no_grad(), autocast('cuda', tc.precision):
    l1 = float(model(x[:, :-1], targets=x[:, 1:]).loss)
print(f"step-1 read on the source weights: loss {l1:.5f} nats ({l1 / 0.6931:.4f} bits/byte)")
model.train(); del x

# the bench: 3 micro-steps at the arm's micro-batch -> the pace -> the hours per arm
opt_probe = torch.optim.SGD(model.parameters(), lr=0.0)
xb = torch.randint(0, 255, (tc.micro_batch, cfg.context), device='cuda')
torch.cuda.reset_peak_memory_stats(); t0 = time.time()
for i in range(3):
    with autocast('cuda', tc.precision):
        out = model(xb, targets=xb)
    out.loss.backward(); opt_probe.zero_grad(set_to_none=True)
torch.cuda.synchronize()
sps = (time.time() - t0) / 3 * tc.grad_accum
vram = torch.cuda.max_memory_allocated() / 2**30
total = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'bench on {torch.cuda.get_device_name(0)} ({total:.0f} GB): {sps:.2f} s/step at {tc.micro_batch} x {tc.grad_accum} · peak {vram:.1f} GB')
assert vram < 0.85 * total, f'over the card budget ({total:.0f} GB): lower MICRO (the step stays 262,144 tokens)'
if HOURS_PER_ARM is None:
    HOURS_PER_ARM = STEPS_PER_ARM * sps / 3600 * 1.05
print(f'HOURS_PER_ARM = {HOURS_PER_ARM:.2f} ({STEPS_PER_ARM} steps + 5%); the queue of {len(ARMS)} arms = {HOURS_PER_ARM * len(ARMS):.1f} h')
del model, out, opt_probe, sd, xb; torch.cuda.empty_cache()
print('PREFLIGHT PASS — C3 is armed')

In [ ]:
# C3 (T) — THE QUEUE. Each arm: a weights-only start from the source checkpoint (the trainer does it when the craft has no
# record on the hub; a later session on the same arm is a normal resume), HOURS_PER_ARM of training, the end report, the
# shipped checkpoint, then the next arm. Interrupt-safe: a stopped arm resumes from its own resume/latest.pt.
import gc, json, time, torch
from geolip.alephllm import prepare
from geolip.alephllm.train import probes
from geolip.alephllm.train.instruments import model_census, toggle_ledger, special_token_gauge
def hf_token():
    import os
    t = os.environ.get('HF_TOKEN')
    if not t:
        from google.colab import userdata
        t = userdata.get('HF_TOKEN')
    assert t, 'no HF token: set HF_TOKEN in the environment (a pod) or in Colab secrets'
    return t

def end_report(run, tag):
    m, tok, dev = run.raw_model, run.tokenizer, run.device
    m.eval()
    with torch.no_grad():
        pr = probes.run_all(m, tok, dev)
        sample = torch.randint(0, 255, (2, min(1024, m.cfg.context)), device=dev)
        census = model_census(m, sample)
        ledger = toggle_ledger(m, run._val())
        sp = special_token_gauge(m, run._val())
    rep = {'tag': tag, 'step': run.step, 'tokens': run.manifest.tokens_seen, 'craft': run.cfg.name,
           'loss_form': run.tc.loss_form, 'loss_copy_beta': run.tc.loss_copy_beta, 'loss_depth_w': list(run.tc.loss_depth_w),
           'init_from': {k: v for k, v in (run.manifest.init_from or {}).items() if k != 'qk_gains'},
           'probes': pr, 'census_flags': census.get('flags'), 'ledger': ledger, 'special': sp}
    run.hub.upload_bytes(json.dumps(rep, default=float).encode(), f'reports/screen/{tag}_step{run.step}.json')
    print(probes.report(pr)); m.train()
    return rep

for PRESET in ARMS:
    print(f"\n===== {PRESET.model.name} =====", flush=True)
    run = prepare(PRESET, hf_token=hf_token())
    if run.manifest.init_from:
        print('start:', json.dumps({k: v for k, v in run.manifest.init_from.items() if k != 'qk_gains'}, indent=1))
    t_end = time.time() + HOURS_PER_ARM * 3600
    while time.time() < t_end:
        hours_left = (t_end - time.time()) / 3600
        if hours_left < 0.05: break
        run.train(max_hours=hours_left, stop_at_boundary=True)
        if getattr(run, '_interrupted', False):
            print('manual stop - no auto-resume; resume state is on the hub'); break
        if run.manifest.current_phase() is None or getattr(run, '_last_boundary', None):
            break
    end_report(run, 'arm_end')
    del run; gc.collect(); torch.cuda.empty_cache()
print('queue over — every arm resumable from its own prefix')

In [ ]:
# C4 — what shipped: each arm's checkpoints and reports on the training repo (the reads run on these).
from huggingface_hub import HfApi
from geolip.alephllm.presets import TRAINING_REPO
api = HfApi()
files = api.list_repo_files(TRAINING_REPO)
for a in ARMS:
    pre = a.model.name
    cks = sorted(f for f in files if f.startswith(f'{pre}/checkpoints/') and f.endswith('.safetensors'))
    reps = sorted(f for f in files if f.startswith(f'{pre}/reports/'))
    print(f"{pre}: {len(cks)} checkpoints (last: {cks[-1].rsplit('/', 1)[-1] if cks else '-'}), {len(reps)} reports")

### Notes
- **One side per session.** `SIDE = 'twin'` runs four arms (about 7 hours on an H100), `SIDE = '2s'` three (about 4 hours).
  A session that ends early leaves the current arm resumable; rerun C1-C3 with the same settings and the queue skips nothing:
  a finished arm simply trains for its remaining hours (none) and re-ships its report.
- **The bed.** `START_STEP` must sit inside the fineweb phase for the rows arms (16,000 or 18,000 for the twin, 18,000 for
  the 2s); the factory refuses a later start. Both twin beds have battery baselines.
- **The reads.** The battery and the probes (reports/byte_battery_2026-10-10/tools on the training repo) run on each arm's
  last checkpoint; the arm's own pass marks are the first byte at 1,024 and 2,048, the lockers above chance, the update
  toward the last value, the digit string's position 12, and held-out bits per byte not worse than `ctrl`.
- **Pace.** C2's bench line is the number to book by; HOURS_PER_ARM follows it unless set by hand.